# Run EXP-07 - BLIP-2 Fusion VQA

Train and evaluate this EXP from the shared mini HDF5 cache copied to local Colab disk.

## 1. Mount Drive and load repo

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = "/content/blip2-fusion-experiment-vqa"
GITHUB_USER = "theflyingkhui04"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/blip2-fusion-experiment-vqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -3

Mounted at /content/drive
Cloning into '/content/blip2-fusion-experiment-vqa'...
remote: Enumerating objects: 339, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 339 (delta 46), reused 62 (delta 40), pack-reused 257 (from 1)
Receiving objects: 100% (339/339), 6.27 MiB | 37.08 MiB/s, done.
Resolving deltas: 100% (175/175), done.
/content/blip2-fusion-experiment-vqa
982f354 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #28 from theflyingkhui04/feat/cdk/template-run
769bed6 (origin/feat/cdk/template-run) Add early-stopping config and mini-cache support
3d34370 Merge pull request #27 from theflyingkhui04/feat/cdk/template-run


## 2. Install dependencies

In [2]:
!pip install -r requirements.txt -q

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

import wandb
wandb.login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 125.5 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
VRAM GB: 15.6


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bluet52hz (bluet52hzzz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 3. Choose run

In [3]:
EXP_ID = "07"
RUN_NUMBER = "2"
YOUR_NAME = "tung"
DATA_ROOT = "/content/drive/MyDrive/blip2_project"

CONFIG_FILE = f"configs/exp{EXP_ID}.yaml"
RUN_NAME = f"exp{EXP_ID}_lan{RUN_NUMBER}_{YOUR_NAME}"
CACHE_DIR_NAME = "cache"
DRIVE_CACHE_DIR = f"{DATA_ROOT}/{CACHE_DIR_NAME}"
LOCAL_CACHE_DIR = "/content/blip2_cache"
ANSWER_LIST = f"{DATA_ROOT}/data/ans2idx.json"
OUTPUT_DIR = f"{DATA_ROOT}/checkpoints/{RUN_NAME}"
EVAL_OUTPUT = f"{OUTPUT_DIR}/val_predictions.json"
CHECKPOINT = f"{OUTPUT_DIR}/best_model.pth"

print("Config:", CONFIG_FILE)
print("Run:", RUN_NAME)
print("Drive mini cache:", DRIVE_CACHE_DIR)
print("Local train cache:", LOCAL_CACHE_DIR)
print("Output:", OUTPUT_DIR)

Config: configs/exp07.yaml
Run: exp07_lan2_tung
Drive mini cache: /content/drive/MyDrive/blip2_project/cache
Local train cache: /content/blip2_cache
Output: /content/drive/MyDrive/blip2_project/checkpoints/exp07_lan2_tung


## 4. Copy mini cache to local disk

In [4]:
import os
import shutil
from pathlib import Path
import h5py

drive_mini = {
    "train": Path(DRIVE_CACHE_DIR) / "train_features_mini.h5",
    "val": Path(DRIVE_CACHE_DIR) / "val_features_mini.h5",
}
local_h5 = {
    "train": Path(LOCAL_CACHE_DIR) / "train_features.h5",
    "val": Path(LOCAL_CACHE_DIR) / "val_features.h5",
}
Path(LOCAL_CACHE_DIR).mkdir(parents=True, exist_ok=True)

def sample_cache(path):
    with h5py.File(path, "r") as f:
        key = next(iter(f.keys()))
        return key, f[key].shape, f[key].dtype

for split, source in drive_mini.items():
    if not source.exists():
        raise FileNotFoundError(f"Missing mini cache artifact: {source}")
    print(split, "Drive sample:", sample_cache(source))
    target = local_h5[split]
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        shutil.copy2(source, target)
    print(split, "Local sample:", sample_cache(target))
print("Question subset sizes come from:", CONFIG_FILE)


train Drive sample: ('100014', (257, 1024), dtype('<f2'))
train Local sample: ('100014', (257, 1024), dtype('<f2'))
val Drive sample: ('100008', (257, 1024), dtype('<f2'))
val Local sample: ('100008', (257, 1024), dtype('<f2'))
Question subset sizes come from: configs/exp07.yaml


## 5. Train

In [ ]:
!python scripts/train.py \
    --config "{CONFIG_FILE}" \
    --run_name "{RUN_NAME}" \
    --data_root "{DATA_ROOT}" \
    --cache_dir "{LOCAL_CACHE_DIR}" \
    --answer_list "{ANSWER_LIST}" \
    --output_dir "{OUTPUT_DIR}"

2026-06-09 15:48:58 | INFO | __main__ | Random seed: 42
2026-06-09 15:48:58 | INFO | __main__ | Sử dụng device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: bluet52hz (bluet52hzzz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run uvzkw382 (0.0s)
wandb: ⣻ setting up run uvzkw382 (0.0s)
wandb: ⣽ setting up run uvzkw382 (0.0s)
wandb: ⣾ setting up run uvzkw382 (0.0s)
wandb: ⣷ setting up run uvzkw382 (0.5s)
wandb: ⣯ setting up run uvzkw382 (0.5s)
wandb: ⣟ setting up run uvzkw382 (0.5s)
wandb: ⡿ setting up run uvzkw382 (0.5s)
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in /content/blip2-fusion-experiment-vqa/wandb/run-20260609_154900-uvzkw382
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run exp07_lan2_tung
wandb: ⭐️ View project at https://wandb.ai/bluet52hzzz/blip2-vqa-experiment
wandb: 🚀 View run at https://wandb.ai/b



```
# Định dạng của đoạn này là mã
```

## 6. Evaluate

In [ ]:
!python scripts/evaluate.py \
    --config "{CONFIG_FILE}" \
    --checkpoint "{CHECKPOINT}" \
    --split val \
    --data_root "{DATA_ROOT}" \
    --cache_dir "{LOCAL_CACHE_DIR}" \
    --answer_list "{ANSWER_LIST}" \
    --output "{EVAL_OUTPUT}"

## 7. Resume

In [ ]:
!python scripts/train.py \
    --config "{CONFIG_FILE}" \
    --run_name "{RUN_NAME}" \
    --data_root "{DATA_ROOT}" \
    --cache_dir "{LOCAL_CACHE_DIR}" \
    --answer_list "{ANSWER_LIST}" \
    --output_dir "{OUTPUT_DIR}" \
    --resume auto